# 03 · Models (RQ1 and RQ3)

All models share the same chronological train/val/test split and the **proposal's reward matrix**
(rows = predicted, columns = true):

| | true easy | true medium | true hard |
|---|---|---|---|
| **pred easy** | +1 | −1 | −3 |
| **pred medium** | 0 | +1 | −1 |
| **pred hard** | −1 | 0 | +1 |

Models
1. **Baselines** – most frequent train class per `(ticker, slot)` (proposal) and a stronger `(ticker, weekday, slot)` + reward-aware variant.
2. **Logistic regression (L2)** – `C` tuned on validation reward; predictions chosen by *expected reward*.
3. **Single-layer softmax trained on the reward loss** – the paper's formulation.
4. **Feedforward net (16, 8, ReLU, dropout, early stopping)** trained on the reward loss (and a cross-entropy variant).

Model selection uses **validation only**; the test set is touched once at the end.

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from htt import config as C

import json
from htt import modeling as M, evaluation as E
ds = pd.read_parquet(C.DATA_PROC / "dataset.parquet")
fam = json.load(open(C.DATA_PROC / "feature_families.json"))

STOCKS = ds[ds.sector.isin(["tech", "fin"])]
tr, va, te = [STOCKS[STOCKS.split == s] for s in ("train", "val", "test")]
cols = fam["all"]
X = lambda d: d[cols].to_numpy(np.float32)
print(len(tr), len(va), len(te), "rows;", len(cols), "features")

## Baselines

In [ ]:
b_slot  = M.HistoricalBaseline(("symbol", "slot")).fit(tr, tr.y)
b_wk    = M.HistoricalBaseline(("symbol", "weekday", "slot")).fit(tr, tr.y)
b_wk_r  = M.HistoricalBaseline(("symbol", "weekday", "slot"), use_reward=True).fit(tr, tr.y)

## Logistic regression

In [ ]:
lr = M.RewardDecisionLR().fit(X(tr), tr.y.values, X(va), va.y.values)
print("best C:", lr.best_C, "| val reward:", round(lr.val_reward, 3))

## Neural models (PyTorch)

In [ ]:
softmax_reward = M.TorchClassifier(hidden=(),       loss="reward", dropout=0.0).fit(X(tr), tr.y.values, X(va), va.y.values)
mlp_reward     = M.TorchClassifier(hidden=(16, 8), loss="reward").fit(X(tr), tr.y.values, X(va), va.y.values)
mlp_ce         = M.TorchClassifier(hidden=(16, 8), loss="ce").fit(X(tr), tr.y.values, X(va), va.y.values)

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, m in [("softmax-reward", softmax_reward), ("MLP-reward", mlp_reward)]:
    h = np.array(m.history); ax.plot(h[:, 1], label=name + " (val)")
ax.set_xlabel("epoch"); ax.set_ylabel("val loss"); ax.legend(); plt.tight_layout()

## Evaluate (validation, then test once)

In [ ]:
def all_preds(d):
    return {
        "baseline_ticker_slot": b_slot.predict(d),
        "baseline_ticker_wkday_slot": b_wk.predict(d),
        "baseline_reward_aware": b_wk_r.predict(d),
        "logreg_L2": lr.predict(X(d)),
        "softmax_reward": softmax_reward.predict(X(d)),
        "mlp_reward": mlp_reward.predict(X(d)),
        "mlp_crossentropy": mlp_ce.predict(X(d)),
    }
val_res  = E.compare(all_preds(va), va.y.values, ref="baseline_ticker_slot")
test_res = E.compare(all_preds(te), te.y.values, ref="baseline_ticker_slot")
print("VALIDATION"); display(val_res.round(3))
print("TEST");       display(test_res.round(3))

**RQ1 (predictability):** do the models beat `baseline_ticker_slot` on `bnr` / `avg_reward` / `macro_f1`?
The `costly_errors` column is the share of rows predicted *easy* that were actually *hard* (the −3 cell).

In [ ]:
test_res.to_csv(C.RESULTS / "test_metrics.csv"); val_res.to_csv(C.RESULTS / "val_metrics.csv")
out = te[["symbol", "sector", "date", "week", "weekday", "slot", "y"]].copy()
for k, v in all_preds(te).items():
    out[k] = v
out.to_parquet(C.RESULTS / "test_predictions.parquet")

## RQ3: is the network worth the loss of interpretability?
Bootstrap over weeks gives a confidence interval on the BNR difference. If the interval includes 0, the
simpler (interpretable) model wins by default.

In [ ]:
P = all_preds(te)
for a, b in [("mlp_reward", "logreg_L2"), ("logreg_L2", "baseline_ticker_slot"), ("mlp_reward", "baseline_ticker_slot")]:
    mean, ci = E.week_bootstrap(te.y.values, P[a], P[b], te.week.values)
    print(f"BNR({a}) - BNR({b}) = {mean:+.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]")